In [ ]:
import cuml
import cupy as cp
import numpy as np
from azure.storage.blob import BlobServiceClient
import io

In [ ]:
# Replace with your actual connection string
account_name="enclaveid"
account_key=
connection_string = f"DefaultEndpointsProtocol=https;AccountName={account_name};AccountKey={account_key};EndpointSuffix=core.windows.net"
container_name = 'enclaveid-dagster-prod-bucket'
blob_name = 'general_interests_clusters/clxenc3fw0007gzz3pdz6enfe.snappy'

# Create the BlobServiceClient object
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create the BlobClient object
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)

# Download the blob content to a bytes buffer
downloaded_blob = blob_client.download_blob()

# Save the blob data to a file
with open("data.snappy", "wb") as file:
    downloaded_blob.readinto(file)

In [ ]:
import polars as pl
df = pl.read_parquet("data.snappy")

In [ ]:
agg_df = df.filter(pl.col("cluster_label") > -1).with_columns(
    (pl.col("date") + ": " + pl.col("interests").cast(str)).alias("date_interest")
).group_by("cluster_label").agg([
    pl.col("date_interest").flatten()
]).with_columns(
    pl.col("date_interest").apply(lambda x: sorted(x)).alias("sorted_date_interest_array")
).drop("date_interest")